### Estructuracion base de datos

Comentario: *NO* se muestran todas las salidas de consultas debido a la sensibilidad de los datos financieros. 
La notebook se deja para demostrar todo el procedimiento de analisis de los datos para la toma de decisiones

Algunas preguntas que se intentan responder a traves de consultas de SQL;
##### Finanzas generales
- Cuanto dinero ingreso?
Cuanto salio
Cual fue el flujo neto?
Como evolucionó el saldo?
##### Gastos
- Cuales son los principales gastos?
- Que proveedores/conceptos se repiten?
- Cuales son los pagos más grandes?

#### Gestión
- Que meses fueron criticos?
- Hay estacionalidad?
- Que gastos son recurrentes?

In [ ]:
# librerias
import pandas as pd
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
print("librerias ok")

librerias ok


In [ ]:
# Cargar variables de conexión desde .env
load_dotenv()
usuario = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
puerto = os.getenv("DB_PORT")
base = os.getenv("DB_NAME")


In [ ]:
# Se conecta con el servidor de postgress que esta local
engine = create_engine(f"postgresql://{usuario}:{password}@{host}:{puerto}/{base}")
# engine.connect()
conn = engine.connect()

In [ ]:
# Se prueba si estamos conectandonos a la base correcta
# Ojo esto abre y ciera la consulta, no la deja abierta por eso hay error despues
with engine.connect() as conn:
    resultado = conn.execute(text("SELECT current_database();"))
    print(resultado.fetchone())

('contabilidad',)


In [ ]:
# Usuario seteado
with engine.connect() as conn:
    resultado = conn.execute(text("SELECT current_user;"))
    print(resultado.fetchone())

('postgres',)


In [ ]:
# Funcion para caragar datos con la conexion abierta de sql
def cargar_tabla(df, nombre_tabla):
    df.to_sql(nombre_tabla,engine, if_exists="append",index=False)
# Funcion para generar consultas con la conexion abierta de sql
def consultar_sql(query):
    return pd.read_sql(query, engine)

In [ ]:
#df con datos locales
df = pd.read_csv("../data/processed/movimientos_bancarios.csv")
#Se cargan datos de los mov bancarios en la tabla que se llama "movimientos bacarios"
cargar_tabla(df, "movimientos_bancarios") ## tienen que coincidir los nombres de las columas exactamente igual

In [ ]:
# Hacemos una primera consulta para ver si funciona o no
consultar_sql("SELECT * FROM movimientos_bancarios LIMIT 0")

,id,fecha,comprobante,movimiento,debito,credito,saldo_en_cuenta,mes,categoria,archivo_origen,fecha_carga


<div align="center">
    <img src="../img/img_consulta_1.png" width="650">
</div>

Con esto comprobamos que:
- El .env está funcionando
- Python lee las credenciales correctas
- SQLAlchemy conecta con PostgreSQL
- La base contabilidad existe
- La tabla movimientos_bancarios existe
- La función consultar_sql() está bien armada

In [ ]:
# Vemos como esta la tabla ahora
consultar_sql("""SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'movimientos_bancarios'
ORDER BY ordinal_position;
""")

,column_name,data_type
0,id,integer
1,fecha,date
2,comprobante,text
3,movimiento,text
4,debito,numeric
5,credito,numeric
6,saldo_en_cuenta,numeric
7,mes,text
8,categoria,text
9,archivo_origen,text


In [ ]:
# Pregunta de negocio: Cuantas operaciones bancarias hubo?

consultar_sql("""SELECT COUNT(*) AS cantidad_registros
FROM movimientos_bancarios
""")

,cantidad_registros
0,4558


In [ ]:
#Vemos el periodo temporal
#Pregunta de negocio: Que historia financiera tenemos cargada?
consultar_sql(""" SELECT MIN(fecha) AS fecha_inicio, MAX(fecha) AS fecha_fin
FROM movimientos_bancarios """)

consultar_sql(""" SELECT MIN(mes) AS mes_inicio, MAX(mes) AS mes_fin
FROM movimientos_bancarios """)

,mes_inicio,mes_fin
0,2022-01,2026-06


In [ ]:
# Revisar movimientos duplicados
consultar_sql("""
SELECT fecha, comprobante, movimiento, debito, credito, COUNT(*) AS cantidad 
FROM movimientos_bancarios
GROUP BY fecha, comprobante, movimiento, debito, credito
HAVING COUNT(*) > 1 
ORDER BY fecha 
LIMIT 0; -- se limitan a mostrar porque los datos son sensibles
""")

<div align="center">
    <img src="../img/imagen_consulta_2.png" width="400">
</div>

In [ ]:
# se revisan valores nulos
consultar_sql("""
SELECT
    COUNT(*) FILTER (WHERE fecha IS NULL) AS fecha_nula,
    COUNT(*) FILTER (WHERE movimiento IS NULL) AS movimiento_nulo,
    COUNT(*) FILTER (WHERE saldo_en_cuenta IS NULL) AS saldo_nulo
FROM movimientos_bancarios;
""")

,fecha_nula,movimiento_nulo,saldo_nulo
0,0,0,0


In [ ]:
# La empresa tuvo flujo positivo o negativo?
consultar_sql("""SELECT
    SUM(credito) AS total_ingresos,
    SUM(debito) AS total_egresos
FROM movimientos_bancarios; """)

In [ ]:
# La empresa tuvo flujo positivo o negativo mes a mes?
consultar_sql("""SELECT mes,
    SUM(credito) AS total_ingresos,
    SUM(debito) AS total_egresos
FROM movimientos_bancarios
GROUP BY mes
ORDER BY mes 
LIMIT 0
; """)

,mes,total_ingresos,total_egresos


Con esto se ven los meses-años de:
- mayor facturacion 
- gastos extraordinarios
- estacionalidad

<div align="center">
    <img src="../img/img_consulta_3.png" width="300">
</div>

In [ ]:
# Flujo neto mensual
consultar_sql("""SELECT mes, SUM(credito)-SUM(debito) AS flujo_neto
FROM movimientos_bancarios
GROUP BY mes
ORDER BY mes
LIMIT 0;""")

,mes,flujo_neto


<div align="center">
    <img src="../img/img_consulta_4.png" width="200">
</div>

In [ ]:
# Analisis por categoria, en que se va el dinero?
consultar_sql(""" SELECT categoria, SUM(debito) AS total_gastado
FROM movimientos_bancarios
GROUP BY categoria
ORDER BY total_gastado DESC
LIMIT 0
; """ )

,categoria,total_gastado


<div align="center">
    <img src="../img/img_consulta_5.png" width="200">
</div>

In [ ]:
# Ranking de categorías de ingreso
consultar_sql(""" SELECT categoria, SUM(credito) AS total_ingresos
FROM movimientos_bancarios
GROUP BY categoria
ORDER BY total_ingresos DESC
LIMIT 0
;""")

,categoria,total_ingresos


<div align="center">
    <img src="../img/img_consulta_6.png" width="200">
</div>

In [ ]:
# Cuales son los movimientos recurrentes
consultar_sql(""" SELECT movimiento, COUNT(*) AS cantidad
FROM movimientos_bancarios
GROUP BY movimiento
ORDER BY cantidad DESC
LIMIT 0;""")

,movimiento,cantidad


<div align="center">
    <img src="../img/img_consulta_7.png" width="250">
</div>

Con esta info se puede detectar:
- proveedores recurrentes
- impuestos
- servicios
- transferencias internas

In [ ]:
# Mayores egresos individuales
consultar_sql ("""SELECT fecha, movimiento, debito
FROM movimientos_bancarios
ORDER BY debito DESC
;""")

In [ ]:
# Saldo minimo y maximo: hubo momentos criticos de caja?. Sii varios
consultar_sql (""" SELECT  MIN(saldo_en_cuenta) AS saldo_minimo, MAX(saldo_en_cuenta) AS saldo_maximo
FROM movimientos_bancarios;""")

In [ ]:
#Saldo al cierre de cada mes
consultar_sql (""" SELECT DISTINCT ON (mes), mes, fecha, saldo_en_cuenta
FROM movimientos_bancarios
ORDER BY mes, fecha DESC; """)

In [ ]:
# Pocos conceptos concentran la mayoría del gasto?
consultar_sql (""" SELECT categoria, SUM(debito) AS gasto, SUM(debito) / (SELECT SUM(debito) FROM movimientos_bancarios) * 100 AS porcentaje
FROM movimientos_bancarios
GROUP BY categoria
ORDER BY porcentaje DESC; """)